# Phase 2 Automated Evaluation

## 1. Objective

Run deterministic offline configuration sweeps against the frozen CISG benchmark. Business logic remains in `src/cial_knowledge_os`; this notebook only orchestrates and displays artifacts.

## 2. Theory

Each configuration receives the same immutable questions. The framework scores keyword coverage, safe failure, citations, hallucination risk, and latency, then ranks the quality/latency trade-off.

## 3. Architecture

`Benchmark → ExperimentGrid → indexed Phase2 pipeline → experiment CSVs → summary/recommendations → standalone dashboard.html`

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'pyproject.toml').is_file()), cwd)
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from cial_knowledge_os import (
    ExperimentGrid, ExperimentRunner, Phase2Config, Phase2RAGPipeline,
    ReconfiguringPipelineFactory, load_benchmark,
)

BENCHMARK_DIR = PROJECT_ROOT / 'data' / 'benchmarks' / 'cisg'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'batch_answers' / '02_Query_Transformations_and_Context_Construction'
benchmark = load_benchmark(
    BENCHMARK_DIR / 'benchmark_answers.csv',
    metadata_path=BENCHMARK_DIR / 'benchmark_metadata.json',
)
len(benchmark.questions)

## 4. Implementation

Index once, then reuse the same local pipeline across configurations. Set `SMOKE_TEST = False` only when the local model and full run budget are ready.

In [ ]:
config = Phase2Config(project_root=PROJECT_ROOT)
pipeline = Phase2RAGPipeline(config)
pipeline.load()
pipeline.chunk()
pipeline.embed()
pipeline.index()

SMOKE_TEST = True

smoke_benchmark = benchmark[:10] if SMOKE_TEST else benchmark

grid = ExperimentGrid({
    'retrieval_top_k': [10] if SMOKE_TEST else [3, 5, 10, 15, 20],
    'max_context_chars': [12000] if SMOKE_TEST else [3000, 6000, 12000, 20000],
    'neighbor_window': [1] if SMOKE_TEST else [0, 1, 2],
    'enable_multi_query': [True] if SMOKE_TEST else [True, False],
    'enable_neighbor_expansion': [True] if SMOKE_TEST else [True, False],
})
'''
grid = ExperimentGrid({
    'retrieval_top_k': [3, 10] if SMOKE_TEST else [3, 5, 10, 15, 20],
    'max_context_chars': [6000] if SMOKE_TEST else [3000, 6000, 12000, 20000],
    'neighbor_window': [0, 1] if SMOKE_TEST else [0, 1, 2],
    'enable_multi_query': [True] if SMOKE_TEST else [True, False],
    'enable_neighbor_expansion': [True] if SMOKE_TEST else [True, False],
})
'''
runner = ExperimentRunner(
    pipeline_factory=ReconfiguringPipelineFactory(pipeline),
    benchmark=smoke_benchmark,
    output_root=OUTPUT_ROOT,
)
result = runner.run(grid)
result.dashboard_file

## 5. Visualization

In [ ]:
from IPython.display import HTML, display
display(HTML(f'<a href="{result.dashboard_file.as_uri()}" target="_blank">Open standalone evaluation dashboard</a>'))

## 6. Benchmark

In [ ]:
import pandas as pd
summary = pd.read_csv(result.summary_file)
display(summary.sort_values('rank'))
print(result.recommendation_file.read_text(encoding='utf-8'))

## 7. Advantages

- Fully offline and repeatable.
- One index build per sweep.
- Per-configuration checkpoints and a portable HTML report.

## 8. Limitations

Keyword evaluation is deterministic but cannot prove semantic entailment. Full Cartesian sweeps require substantial local generation time.

## 9. Enterprise Considerations

All questions, context, answers, traces, and reports stay on-premise. CSV artifacts retain inspectable evidence for audit and regression review.

## 10. What we'll improve in the next notebook

Phase 3 can register hybrid-retrieval recall, lexical/vector contribution, and reranking metrics through the same runner and dashboard artifact contract.